In [ ]:
# env-base(installed at base)
import scanpy as sc
import torch
import scarches as sca
print(sca.__version__)
from scarches.dataset.trvae.data_handling import remove_sparsity
import matplotlib.pyplot as plt
import numpy as np
import gdown
from anndata import AnnData
import numpy as np
import pandas as pd
import os
sc.settings.seed = 42
sc.settings.set_figure_params(dpi=200, frameon=False)
sc.set_figure_params(dpi=200)
sc.set_figure_params(figsize=(4, 4))
torch.set_printoptions(precision=3, sci_mode=False, edgeitems=7)

In [2]:
##load dataset
adata = sc.read_h5ad('/data2/lanxiang/embryo_benchmark/25.1.13_update/human_clustering_20250724_v3.h5ad')

In [ ]:
adata

In [ ]:
adata.obs

In [ ]:
sc.pl.umap(adata, color=['reanno'])
sc.pl.umap(adata, color=['lineage'])

In [6]:
adata_hvg = adata[:, adata.var.highly_variable].copy()

In [ ]:
adata_hvg

In [ ]:
import anndata

print("Original obs:", adata_hvg.obs.columns)
print("Original var:", adata_hvg.var.columns)
print("Original uns:", adata_hvg.uns.keys())
print("Original obsm:", adata_hvg.obsm.keys())
print("Original varm:", adata_hvg.varm.keys())
print("Original layers:", adata_hvg.layers.keys())

obs_to_keep = ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'stage', 'percent.mt', 'species', 'embryo', 'platform', 'ann_level_2', 'ann_level_3', 'ann_level_1', 'doublet', 'doublet_score', 'scANVI_res_0.5', 'leiden_3', 'reanno', 'combined_annotation', 'unicorns', 'lineage']
obs_columns_to_remove = [col for col in adata_hvg.obs.columns if col not in obs_to_keep]
adata_hvg.obs.drop(columns=obs_columns_to_remove, inplace=True)

features_to_keep = ['features']
var_columns_to_remove = [col for col in adata_hvg.var.columns if col not in features_to_keep]
adata_hvg.var.drop(columns=var_columns_to_remove, inplace=True)

adata_hvg.uns = {}

adata_hvg.obsm = {}

adata_hvg.varm = {}

layers_to_keep = ['counts']
layers_keys_to_remove = [key for key in adata_hvg.layers.keys() if key not in layers_to_keep]
for key in layers_keys_to_remove:
    del adata_hvg.layers[key]

print("Modified obs:", adata_hvg.obs.columns)
print("Modified var:", adata_hvg.var.columns)
print("Modified uns:", adata_hvg.uns.keys())
print("Modified obsm:", adata_hvg.obsm.keys())
print("Modified varm:", adata_hvg.varm.keys())
print("Modified layers:", adata_hvg.layers.keys())

In [9]:
counts_matrix = adata_hvg.layers["counts"].toarray()
adata = sc.AnnData(X=counts_matrix, obs=adata_hvg.obs.copy(), var=adata_hvg.var.copy(),uns=adata_hvg.uns.copy(), obsm = adata_hvg.obsm.copy(),varm = adata_hvg.varm.copy(),layers={'counts': counts_matrix})

In [10]:
adata

AnnData object with n_obs × n_vars = 33794 × 2000
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'stage', 'percent.mt', 'species', 'embryo', 'platform', 'doublet', 'doublet_score', 'scANVI_res_0.5', 'leiden_3', 'reanno', 'combined_annotation', 'unicorns', 'lineage'
    var: 'features'
    layers: 'counts'

In [11]:
adata = remove_sparsity(adata)

In [12]:
source_adata = adata.copy()

In [ ]:
Weatherbee = sc.read_h5ad('/data2/lanxiang/embryo_benchmark/9.27labeltransfer_data/corrected_processed_Weatherbee.h5ad')
Weatherbee

In [ ]:
sc.settings.seed = 42
Weatherbee.layers["counts"] =Weatherbee.X.copy()
sc.pp.normalize_total(Weatherbee, target_sum=1e4)
sc.pp.log1p(Weatherbee)
Weatherbee.layers["logcounts"] = Weatherbee.X.copy()
sc.pp.highly_variable_genes(Weatherbee, n_top_genes=2000, flavor="cell_ranger", batch_key="orig.ident")
sc.tl.pca(Weatherbee, n_comps=30, use_highly_variable=True)

In [ ]:
Weatherbee

In [16]:
counts_matrix = Weatherbee.layers["counts"].toarray()
adata3 = sc.AnnData(X=counts_matrix, obs=Weatherbee.obs.copy(), var=Weatherbee.var.copy(), layers={'counts': counts_matrix})

In [17]:
adata3 = remove_sparsity(adata3)

In [ ]:
adata3

In [19]:
all_genes = source_adata.var_names
missing_genes = all_genes.difference(adata3.var_names)
missing_data = np.zeros((adata3.shape[0], len(missing_genes)))
adata3_df = pd.DataFrame(adata3.X, columns=adata3.var_names, index=adata3.obs_names)
missing_df = pd.DataFrame(missing_data, columns=missing_genes, index=adata3.obs_names)
adata3_combined_df = pd.concat([adata3_df, missing_df], axis=1)
adata3_combined_df = adata3_combined_df[all_genes]
adata3_extended = sc.AnnData(
    X=adata3_combined_df.values, 
    obs=adata3.obs,
    var=pd.DataFrame(index=all_genes),
    layers={'counts': adata3_combined_df.values})
adata3_extended.var['features'] = Weatherbee.var.reindex(all_genes)['features']


In [ ]:
adata3_extended

In [21]:
obs_to_keep = ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'sample_type', 'scmap_nakamura', 'scmapCELL_Yang', 'scmap_ma', 'scmap_Tyser', 'scmapCELL_Mole', 'cell_assignment', 'course_cell_assignment', 'stage', 'species', 'embryo', 'platform']
obs_columns_to_remove = [col for col in adata3_extended.obs.columns if col not in obs_to_keep]
adata3_extended.obs.drop(columns=obs_columns_to_remove, inplace=True)

In [ ]:
print(adata3_extended.obs.dtypes)

In [ ]:
print(source_adata.obs.dtypes)

In [24]:
target_adata = adata3_extended.copy()

In [25]:
original_source_adata=source_adata.copy()
original_target_adata = target_adata.copy()

# final_anno

In [26]:
condition_key = 'orig.ident'
cell_type_key = "reanno"

In [ ]:
sca.models.SCVI.setup_anndata(source_adata, batch_key=condition_key, labels_key=cell_type_key)
vae = sca.models.SCVI(
    source_adata,
    n_layers=2,
    encode_covariates=True,
    deeply_inject_covariates=False,
    use_layer_norm="both",
    use_batch_norm="none",
)
vae.train()

In [28]:
scanvae = sca.models.SCANVI.from_scvi_model(vae, unlabeled_category = "Unknown")
print("Labelled Indices: ", len(scanvae._labeled_indices))
print("Unlabelled Indices: ", len(scanvae._unlabeled_indices))

Labelled Indices:  33794
Unlabelled Indices:  0


In [ ]:
scanvae.train(max_epochs=20)
reference_latent = sc.AnnData(scanvae.get_latent_representation())
reference_latent.obs["cell_type"] = source_adata.obs[cell_type_key].tolist()
reference_latent.obs["batch"] = source_adata.obs[condition_key].tolist()

In [ ]:
sc.pp.neighbors(reference_latent, n_neighbors=10)
sc.tl.leiden(reference_latent)
sc.tl.umap(reference_latent)
sc.pl.umap(reference_latent,
           color=['batch', 'cell_type'],
           frameon=False,
           wspace=0.6,)

In [ ]:
reference_latent.obs['predictions'] = scanvae.predict()
reference_latent.obs

In [32]:
ref_path = '/data2/lanxiang/embryo_benchmark/25.1.13_update/scArches/human_Weatherbee/final_anno_model'
scanvae.save(ref_path, overwrite=True)

In [ ]:
target_adata

In [ ]:
target_adata.obs

In [ ]:
model = sca.models.SCANVI.load_query_data(
    target_adata,
    ref_path,
    freeze_dropout = True,
)

In [ ]:
model._unlabeled_indices = np.arange(target_adata.n_obs)
model._labeled_indices = []
print("Labelled Indices: ", len(model._labeled_indices))
print("Unlabelled Indices: ", len(model._unlabeled_indices))

In [ ]:
model.train(
    max_epochs=100,
    plan_kwargs=dict(weight_decay=0.0),
    check_val_every_n_epoch=10,
)

In [38]:
query_latent = sc.AnnData(model.get_latent_representation())
query_latent.obs['cell_type'] = target_adata.obs[cell_type_key].tolist()
query_latent.obs['batch'] = target_adata.obs[condition_key].tolist()

In [39]:
sc.pp.neighbors(query_latent)
sc.tl.leiden(query_latent)
sc.tl.umap(query_latent)

In [40]:
surg_path = '/data2/lanxiang/embryo_benchmark/25.1.13_update/scArches/human_Weatherbee/final_anno_surg_model'
model.save(surg_path, overwrite=True)

In [41]:
query_latent.obs['predictions'] = model.predict()
print("Acc: {}".format(np.mean(query_latent.obs.predictions == query_latent.obs.cell_type)))

Acc: 0.0


In [ ]:
adata_full = source_adata.concatenate(target_adata)
adata_full.obs["is_ref"] = ["Query"] * len(target_adata) + ["Reference"] * len(
    source_adata
)

In [ ]:
adata_full

In [ ]:
adata_full.obs

In [ ]:
full_latent = sc.AnnData(model.get_latent_representation(adata=adata_full))
full_latent.obs['cell_type'] = adata_full.obs[cell_type_key].tolist()
full_latent.obs['batch'] = adata_full.obs[condition_key].tolist()

In [ ]:
full_latent

In [ ]:
full_latent.obs

In [ ]:
sc.pp.neighbors(full_latent)
sc.tl.leiden(full_latent)
sc.tl.umap(full_latent)
plt.figure()
sc.pl.umap(
    full_latent,
    color=["batch", "cell_type"],
    frameon=False,
    wspace=0.6,
)

In [ ]:
full_latent.obs['predictions'] = model.predict(adata=adata_full)
print("Acc: {}".format(np.mean(full_latent.obs.predictions == full_latent.obs.cell_type)))

In [ ]:
full_latent

In [ ]:
full_latent.obs

In [ ]:
sc.pp.neighbors(full_latent)
sc.tl.leiden(full_latent)
sc.tl.umap(full_latent)
plt.figure()
sc.pl.umap(
    full_latent,
    color=["predictions", "cell_type"],
    frameon=False,
    wspace=0.6,
)

In [53]:
adata_full.obs["scArches_final_anno_pre"] = full_latent.obs["predictions"].values

In [ ]:
adata_full.obs

In [ ]:
knn_transformer = sca.utils.knn.weighted_knn_trainer(
    train_adata=reference_latent,
    train_adata_emb="X",  # Use X from the latent
    n_neighbors=50,       
)

labels, uncert = sca.utils.knn.weighted_knn_transfer(
    query_adata=query_latent, 
    query_adata_emb="X", 
    label_keys="predictions",  # use SCANVI predicted labels
    knn_model=knn_transformer,
    ref_adata_obs=reference_latent.obs,  
)

In [ ]:
uncert

In [57]:
Weatherbee_data = adata_full.obs[adata_full.obs["orig.ident"] == "Weatherbee"]

In [ ]:
Weatherbee_data

In [ ]:
Weatherbee.obs

In [60]:
Weatherbee.obs["scArches_final_anno_pre"] = Weatherbee_data["scArches_final_anno_pre"].values
Weatherbee.obs["scArches_final_anno_uncertainty"] = uncert["predictions"].values

In [ ]:
Weatherbee.obs

# final_lineage

In [62]:
condition_key = 'orig.ident'
cell_type_key = "lineage"

In [63]:
source_adata=original_source_adata.copy()
target_adata=original_target_adata.copy()

In [ ]:
sca.models.SCVI.setup_anndata(source_adata, batch_key=condition_key, labels_key=cell_type_key)
vae = sca.models.SCVI(
    source_adata,
    n_layers=2,
    encode_covariates=True,
    deeply_inject_covariates=False,
    use_layer_norm="both",
    use_batch_norm="none",
)
vae.train()

In [ ]:
scanvae = sca.models.SCANVI.from_scvi_model(vae, unlabeled_category = "Unknown")
print("Labelled Indices: ", len(scanvae._labeled_indices))
print("Unlabelled Indices: ", len(scanvae._unlabeled_indices))

In [ ]:
scanvae.train(max_epochs=20)
reference_latent = sc.AnnData(scanvae.get_latent_representation())
reference_latent.obs["cell_type"] = source_adata.obs[cell_type_key].tolist()
reference_latent.obs["batch"] = source_adata.obs[condition_key].tolist()

In [ ]:
sc.pp.neighbors(reference_latent, n_neighbors=10)
sc.tl.leiden(reference_latent)
sc.tl.umap(reference_latent)
sc.pl.umap(reference_latent,
           color=['batch', 'cell_type'],
           frameon=False,
           wspace=0.6,)

In [ ]:
reference_latent.obs['predictions'] = scanvae.predict()
reference_latent.obs

In [69]:
ref_path = '/data2/lanxiang/embryo_benchmark/25.1.13_update/scArches/human_Weatherbee/final_lineage_model'
scanvae.save(ref_path, overwrite=True)

In [ ]:
target_adata

In [ ]:
target_adata.obs

In [ ]:
model = sca.models.SCANVI.load_query_data(
    target_adata,
    ref_path,
    freeze_dropout = True,
)

In [ ]:
model._unlabeled_indices = np.arange(target_adata.n_obs)
model._labeled_indices = []
print("Labelled Indices: ", len(model._labeled_indices))
print("Unlabelled Indices: ", len(model._unlabeled_indices))

In [ ]:
model.train(
    max_epochs=100,
    plan_kwargs=dict(weight_decay=0.0),
    check_val_every_n_epoch=10,
)

In [75]:
query_latent = sc.AnnData(model.get_latent_representation())
query_latent.obs['cell_type'] = target_adata.obs[cell_type_key].tolist()
query_latent.obs['batch'] = target_adata.obs[condition_key].tolist()

In [76]:
sc.pp.neighbors(query_latent)
sc.tl.leiden(query_latent)
sc.tl.umap(query_latent)

In [77]:
surg_path = '/data2/lanxiang/embryo_benchmark/25.1.13_update/scArches/human_Weatherbee/final_lineage_surg_model'
model.save(surg_path, overwrite=True)

In [78]:
query_latent.obs['predictions'] = model.predict()

In [ ]:
adata_full = source_adata.concatenate(target_adata)
adata_full.obs["is_ref"] = ["Query"] * len(target_adata) + ["Reference"] * len(
    source_adata
)

In [ ]:
adata_full

In [ ]:
adata_full.obs

In [ ]:
full_latent = sc.AnnData(model.get_latent_representation(adata=adata_full))
full_latent.obs['cell_type'] = adata_full.obs[cell_type_key].tolist()
full_latent.obs['batch'] = adata_full.obs[condition_key].tolist()

In [ ]:
sc.pp.neighbors(full_latent)
sc.tl.leiden(full_latent)
sc.tl.umap(full_latent)
plt.figure()
sc.pl.umap(
    full_latent,
    color=["batch", "cell_type"],
    frameon=False,
    wspace=0.6,
)

In [ ]:
full_latent.obs['predictions'] = model.predict(adata=adata_full)
print("Acc: {}".format(np.mean(full_latent.obs.predictions == full_latent.obs.cell_type)))

In [ ]:
full_latent

In [ ]:
full_latent.obs

In [ ]:
sc.pp.neighbors(full_latent)
sc.tl.leiden(full_latent)
sc.tl.umap(full_latent)
plt.figure()
sc.pl.umap(
    full_latent,
    color=["predictions", "cell_type"],
    frameon=False,
    wspace=0.6,
)

In [88]:
adata_full.obs["scArches_final_lineage_pre"] = full_latent.obs["predictions"].values

In [ ]:
adata_full.obs

In [ ]:
knn_transformer = sca.utils.knn.weighted_knn_trainer(
    train_adata=reference_latent,
    train_adata_emb="X", 
    n_neighbors=50,      
)

labels, uncert = sca.utils.knn.weighted_knn_transfer(
    query_adata=query_latent, 
    query_adata_emb="X", 
    label_keys="predictions",
    knn_model=knn_transformer,
    ref_adata_obs=reference_latent.obs,  
)

In [ ]:
uncert

In [92]:
Weatherbee_data = adata_full.obs[adata_full.obs["orig.ident"] == "Weatherbee"]

In [93]:
Weatherbee.obs["scArches_final_lineage_pre"] = Weatherbee_data["scArches_final_lineage_pre"].values
Weatherbee.obs["scArches_final_lineage_uncertainty"] = uncert["predictions"].values

In [ ]:
Weatherbee.obs

In [95]:
Weatherbee.obs["scArches_final_anno_uncertainty"] = Weatherbee.obs["scArches_final_anno_uncertainty"].astype(str)
Weatherbee.obs["scArches_final_lineage_uncertainty"] = Weatherbee.obs["scArches_final_lineage_uncertainty"].astype(str)

In [96]:
Weatherbee.obs.to_csv("/data2/lanxiang/embryo_benchmark/25.1.13_update/embryo_model_csv_updata/human_Weatherbee_scArches.csv", index=True)
Weatherbee.write_h5ad("/data2/lanxiang/embryo_benchmark/25.1.13_update/embryo_model_csv_updata/human_Weatherbee_scArches.h5ad")